In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
train = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/Hotel Chain A - Supporting Resources/Hotel-A-train.csv')
val   = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/Hotel Chain A - Supporting Resources/Hotel-A-validation.csv')
test  = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/Hotel Chain A - Supporting Resources/Hotel-A-test.csv')

In [3]:
print('Train shape     :', train.shape)
print('Validation shape:', val.shape)
print('Test shape      :', test.shape)

Train shape     : (27499, 24)
Validation shape: (2749, 24)
Test shape      : (4318, 23)


## Missing values

In [4]:
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    print(f'\n{name}:')
    missing = df.isnull().sum()
    pct = (missing / len(df)*100.0).round(2)
    summary = pd.DataFrame({'Missing': missing, 'Pct(%)': pct})
    print(summary[summary['Missing'] > 0].to_string() or 'No missing values')


TRAIN:
Empty DataFrame
Columns: [Missing, Pct(%)]
Index: []

VALIDATION:
Empty DataFrame
Columns: [Missing, Pct(%)]
Index: []

TEST:
Empty DataFrame
Columns: [Missing, Pct(%)]
Index: []


## Check Duplicates

In [5]:
for name, df in [('Train', train), ('Validation', val), ('Testgi', test)]:
    print(f'{name} duplicates: {df.duplicated().sum()}')

Train duplicates: 0
Validation duplicates: 0
Testgi duplicates: 0


## Convert Date Columns

In [6]:
date_cols = ['Expected_checkin', 'Expected_checkout', 'Booking_date']

for df in [train, val, test]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Verify
print(train[date_cols].dtypes)

Expected_checkin     datetime64[ns]
Expected_checkout    datetime64[ns]
Booking_date         datetime64[ns]
dtype: object


## Fix Reservation_status casing

In [7]:
# Before fix
print('BEFORE FIX:')
print('TRAIN:', train['Reservation_Status'].value_counts())
print('VAL  :', val['Reservation_Status'].value_counts())

# Apply fix
train['Reservation_Status'] = (train['Reservation_Status']
                                .str.strip()
                                .replace({'Check-out': 'Check-Out',
                                          'Canceled' : 'Cancelled'}))

val['Reservation_Status'] = (val['Reservation_Status']
                              .str.strip()
                              .replace({'Check-In' : 'Check-Out',
                                        'Canceled' : 'Cancelled'}))

# After fix
print('\nAFTER FIX:')
print('TRAIN:', train['Reservation_Status'].value_counts())
print('VAL  :', val['Reservation_Status'].value_counts())

BEFORE FIX:
TRAIN: Reservation_Status
Check-Out    19092
Canceled      4134
Check-out     2148
No-Show       2125
Name: count, dtype: int64
VAL  : Reservation_Status
Check-In    1610
Canceled     741
No-Show      398
Name: count, dtype: int64

AFTER FIX:
TRAIN: Reservation_Status
Check-Out    21240
Cancelled     4134
No-Show       2125
Name: count, dtype: int64
VAL  : Reservation_Status
Check-Out    1610
Cancelled     741
No-Show       398
Name: count, dtype: int64


## Encode Reservation_Status numerically

In [8]:
status_map = {'Check-Out': 1, 'Cancelled': 2, 'No-Show': 3}

train['Reservation_Status_Code'] = train['Reservation_Status'].map(status_map)
val['Reservation_Status_Code']   = val['Reservation_Status'].map(status_map)

# Verify — should show no NaN
print('TRAIN NaN in status code:', train['Reservation_Status_Code'].isnull().sum())
print('VAL   NaN in status code:', val['Reservation_Status_Code'].isnull().sum())

TRAIN NaN in status code: 0
VAL   NaN in status code: 0


## Create binary target Variables

In [9]:
train['Cancelled'] = (train['Reservation_Status'] == 'Cancelled').astype(int)
train['No_Show']   = (train['Reservation_Status'] == 'No-Show').astype(int)
val['Cancelled']   = (val['Reservation_Status'] == 'Cancelled').astype(int)
val['No_Show']     = (val['Reservation_Status'] == 'No-Show').astype(int)

# Verify
print('TRAIN targets:')
print(train.groupby('Reservation_Status')[['Cancelled', 'No_Show']].sum())

print('\nVAL targets:')
print(val.groupby('Reservation_Status')[['Cancelled', 'No_Show']].sum())

TRAIN targets:
                    Cancelled  No_Show
Reservation_Status                    
Cancelled                4134        0
Check-Out                   0        0
No-Show                     0     2125

VAL targets:
                    Cancelled  No_Show
Reservation_Status                    
Cancelled                 741        0
Check-Out                   0        0
No-Show                     0      398


## Check and fix capacity violations

In [10]:
# Check before fix
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    violations = df[df['Adults'] + df['Children'] > 5]
    print(f'{name} — violations before fix: {len(violations)}')

# Apply fix — cap Children so Adults + Children never exceeds 5
for df in [train, val, test]:
    df['Children'] = df.apply(lambda row: min(row['Children'], 5 - row['Adults']), axis=1)

# Verify
print()
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    violations = df[df['Adults'] + df['Children'] > 5]
    print(f'{name} — violations after fix: {len(violations)}')

TRAIN — violations before fix: 4369
VALIDATION — violations before fix: 422
TEST — violations before fix: 727

TRAIN — violations after fix: 0
VALIDATION — violations after fix: 0
TEST — violations after fix: 0


## Inconsistent data

In [11]:
# Age range Check
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    invalid_age = df[(df['Age'] < 18) | (df['Age'] > 70)]
    print(f'{name} — Age out of range (< 18 or > 70): {len(invalid_age)}')

TRAIN — Age out of range (< 18 or > 70): 0
VALIDATION — Age out of range (< 18 or > 70): 0
TEST — Age out of range (< 18 or > 70): 0


In [12]:
# check negative lead times

date_cols = ['Expected_checkin', 'Expected_checkout', 'Booking_date']
for df in [train, val, test]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Check
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    negative_lead = df[df['Expected_checkin'] < df['Booking_date']]
    print(f'{name} — Negative lead time (check-in before booking): {len(negative_lead)}')

TRAIN — Negative lead time (check-in before booking): 506
VALIDATION — Negative lead time (check-in before booking): 16
TEST — Negative lead time (check-in before booking): 27


In [13]:
# Invalide stay duration (check-out before check-in)

for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    invalid_stay = df[df['Expected_checkout'] <= df['Expected_checkin']]
    print(f'{name} — Invalid stay duration (checkout <= check-in): {len(invalid_stay)}')

TRAIN — Invalid stay duration (checkout <= check-in): 0
VALIDATION — Invalid stay duration (checkout <= check-in): 0
TEST — Invalid stay duration (checkout <= check-in): 0


# Outliers

In [14]:
num_cols = ['Age', 'Adults', 'Children', 'Babies', 'Discount_Rate', 'Room_Rate']

print('Outlier counts per column (Train):')
for col in num_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = train[(train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 + 1.5 * IQR)]
    print(f'  {col:<20} — outliers: {len(outliers)}')

Outlier counts per column (Train):
  Age                  — outliers: 0
  Adults               — outliers: 2286
  Children             — outliers: 0
  Babies               — outliers: 0
  Discount_Rate        — outliers: 0
  Room_Rate            — outliers: 0


In [15]:
# Fix — cap outliers using IQR bounds (Winsorization)
for df in [train, val, test]:
    for col in num_cols:
        Q1  = train[col].quantile(0.25)
        Q3  = train[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower=lower, upper=upper)

# Verify
print('Outlier counts after fix (Train):')
for col in num_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = train[(train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 + 1.5 * IQR)]
    print(f'  {col:<20} — outliers: {len(outliers)}')

Outlier counts after fix (Train):
  Age                  — outliers: 0
  Adults               — outliers: 0
  Children             — outliers: 0
  Babies               — outliers: 0
  Discount_Rate        — outliers: 0
  Room_Rate            — outliers: 0


In [16]:
for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    print(f'\n{name}')
    print(f'  Shape          : {df.shape}')
    print(f'  Missing values : {df.isnull().sum().sum()}')
    print(f'  Duplicates     : {df.duplicated().sum()}')


TRAIN
  Shape          : (27499, 27)
  Missing values : 0
  Duplicates     : 0

VALIDATION
  Shape          : (2749, 27)
  Missing values : 0
  Duplicates     : 0

TEST
  Shape          : (4318, 23)
  Missing values : 0
  Duplicates     : 0


In [20]:
train.head()

,Reservation-id,Gender,Age,Ethnicity,Educational_Level,Income,Country_region,Hotel_Type,Expected_checkin,Expected_checkout,...,Deposit_type,Booking_channel,Required_Car_Parking,Reservation_Status,Use_Promotion,Discount_Rate,Room_Rate,Reservation_Status_Code,Cancelled,No_Show
0,39428300,F,40,Latino,Grad,<25K,North,City Hotel,2015-07-01,2015-07-02,...,No Deposit,Online,Yes,Check-Out,Yes,10,218,1,0,0
1,77491756,F,49,Latino,Mid-School,50K -- 100K,East,City Hotel,2015-07-01,2015-07-02,...,Refundable,Online,Yes,Check-Out,No,0,185,1,0,0
2,73747291,F,42,caucasian,Grad,<25K,East,City Hotel,2015-07-02,2015-07-06,...,No Deposit,Online,Yes,Check-Out,No,0,119,1,0,0
3,67301739,M,25,African American,College,>100K,South,Airport Hotels,2015-07-02,2015-07-03,...,Refundable,Agent,Yes,Check-Out,Yes,5,144,1,0,0
4,77222321,F,62,Latino,High-School,25K --50K,East,Resort,2015-07-03,2015-07-04,...,No Deposit,Direct,No,Check-Out,Yes,10,242,1,0,0


In [21]:
val.head()

,Reservation-id,Gender,Age,Ethnicity,Educational_Level,Income,Country_region,Hotel_Type,Expected_checkin,Expected_checkout,...,Deposit_type,Booking_channel,Required_Car_Parking,Reservation_Status,Use_Promotion,Discount_Rate,Room_Rate,Reservation_Status_Code,Cancelled,No_Show
0,45716350,M,56,caucasian,Grad,<25K,West,Resort,2016-08-31,2016-09-02,...,No Deposit,Agent,No,No-Show,Yes,15,192,3,0,1
1,88857401,M,60,Latino,College,25K --50K,West,Resort,2016-08-31,2016-09-04,...,No Deposit,Online,Yes,Cancelled,No,0,187,2,1,0
2,16074440,F,58,Asian American,College,<25K,North,Airport Hotels,2016-09-01,2016-09-02,...,No Deposit,Direct,No,Cancelled,Yes,10,227,2,1,0
3,10992124,F,23,Latino,College,25K --50K,East,Airport Hotels,2016-08-31,2016-09-02,...,Refundable,Direct,No,Check-Out,Yes,25,189,1,0,0
4,15934351,F,47,Asian American,College,25K --50K,South,City Hotel,2016-08-31,2016-09-01,...,No Deposit,Online,Yes,Check-Out,Yes,10,218,1,0,0


In [22]:
test.head()

,Reservation-id,Gender,Age,Ethnicity,Educational_Level,Income,Country_region,Hotel_Type,Expected_checkin,Expected_checkout,...,Babies,Meal_Type,Visted_Previously,Previous_Cancellations,Deposit_type,Booking_channel,Required_Car_Parking,Use_Promotion,Discount_Rate,Room_Rate
0,62931593,F,52,Latino,Grad,25K --50K,South,City Hotel,2016-11-18,2016-11-19,...,0,HB,No,No,No Deposit,Direct,Yes,Yes,10,153
1,70586099,F,47,Latino,Grad,25K --50K,East,Airport Hotels,2016-11-18,2016-11-19,...,0,FB,No,No,No Deposit,Online,No,No,0,210
2,4230648,F,28,Asian American,Grad,<25K,East,City Hotel,2017-04-28,2017-05-01,...,0,BB,No,No,No Deposit,Agent,No,Yes,5,117
3,25192322,F,65,caucasian,High-School,25K --50K,South,Airport Hotels,2016-11-18,2016-11-20,...,2,FB,No,No,No Deposit,Online,Yes,Yes,10,107
4,80931528,M,45,African American,College,25K --50K,South,City Hotel,2016-11-18,2016-11-20,...,0,BB,No,No,Refundable,Agent,No,No,0,119


In [ ]:
#Create folder
os.makedirs('cleaned_dataset', exist_ok=True)

# Save cleaned datasets
train.to_csv('cleaned_dataset/train_cleaned.csv', index=False)
val.to_csv('cleaned_dataset/val_cleaned.csv',     index=False)
test.to_csv('cleaned_dataset/test_cleaned.csv',   index=False)

print('Cleaned datasets saved to cleaned_dataset folder.')

Cleaned datasets saved to cleaned_dataset folder.
